## Загрузка и анализ датасета

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
df = pd.read_csv('../data/SMSSpamCollection', sep='\t', header=None, names=['target', 'text'], encoding='utf-8')

In [5]:
df.shape

(5572, 2)

In [6]:
df['target'].value_counts(normalize=True)

target
ham     0.865937
spam    0.134063
Name: proportion, dtype: float64

Видим сильный дисбаланс классов

In [7]:
df['length'] = df['text'].apply(len)

In [8]:
df['length'].describe()

count    5572.000000
mean       80.489950
std        59.942907
min         2.000000
25%        36.000000
50%        62.000000
75%       122.000000
max       910.000000
Name: length, dtype: float64

In [10]:
df[df['target'] == 'ham'].head()

,target,text,length
0,ham,"Go until jurong point, crazy.. Available only ...",111
1,ham,Ok lar... Joking wif u oni...,29
3,ham,U dun say so early hor... U c already then say...,49
4,ham,"Nah I don't think he goes to usf, he lives aro...",61
6,ham,Even my brother is not like to speak with me. ...,77


In [15]:
df[df['target'] == 'spam']['text'][2]

"Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"

In [9]:
df['target'] = df['target'].map({'ham': 0, 'spam': 1})

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['target'],
    test_size=0.2,
    random_state=42,
    stratify=df['target']
)

In [11]:
X_train

184                              He will, you guys close?
2171    CAN I PLEASE COME UP NOW IMIN TOWN.DONTMATTER ...
5422              Ok k..sry i knw 2 siva..tats y i askd..
4113                            I'll see, but prolly yeah
4588    I'll see if I can swing by in a bit, got some ...
                              ...                        
1932                  What pa tell me.. I went to bath:-)
5316                         Jus finish watching tv... U?
2309    Moby Pub Quiz.Win a £100 High Street prize if ...
1904    Free entry in 2 a weekly comp for a chance to ...
762     We are at grandmas. Oh dear, u still ill? I fe...
Name: text, Length: 4457, dtype: object

In [12]:
y_train

184     0
2171    0
5422    0
4113    0
4588    0
       ..
1932    0
5316    0
2309    1
1904    1
762     0
Name: target, Length: 4457, dtype: int64

## Baseline TF-IDF + LogisticRegression

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

In [14]:
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [15]:
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [16]:
y_pred = model.predict(X_test_tfidf)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred))
print('Recall:', recall_score(y_test, y_pred))
print('F1-score:', f1_score(y_test, y_pred))

Accuracy: 0.9730941704035875
Precision: 1.0
Recall: 0.7986577181208053
F1-score: 0.8880597014925373


Видим 100% precision, однако модель пропускает около 18% спама. Посмотрим на примеры пропущеных сообщений

In [17]:
for i in range(20):
    print(i,':',X_test[y_pred == y_test].iloc[i])

0 : FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv
1 : Dear Voucher Holder 2 claim your 1st class airport lounge passes when using Your holiday voucher call 08704439680. When booking quote 1st class x 2
2 : Talk sexy!! Make new friends or fall in love in the worlds most discreet text dating service. Just text VIP to 83110 and see who you could meet.
3 : Hi if ur lookin 4 saucy daytime fun wiv busty married woman Am free all next week Chat now 2 sort time 09099726429 JANINExx Calls£1/minMobsmoreLKPOBOX177HP51FL
4 : ringtoneking 84484
5 : Sorry I missed your call let's talk when you have the time. I'm on 07090201529
6 : Rock yr chik. Get 100's of filthy films &XXX pics on yr phone now. rply FILTH to 69669. Saristar Ltd, E14 9YT 08701752560. 450p per 5 days. Stop2 cancel
7 : Latest News! Police station toilet stolen, cops have nothing to go on!
8 : Sunshine Quiz! Win a super Sony DVD recor

Ожидается, что BERT будет лучше ловить контекст и recall вырастет